# Pretrain the language model with the own engine on Kaggle (GPU)

Builds `wav-transformer` with `--features gpu-engine`: hand-written forward and backward pass, cuBLAS GEMMs in f16 on
the tensor cores (f32 accumulation, dynamic loss scaling) and own CUDA kernels compiled at run time with NVRTC — no
CUDA toolkit build, no nvcc. Same architecture as before (width 256, 12 layers, attention window 256, ~63 M
parameters), with the complex layer norm on the real variance and the split-SiLU gate.

**Once:** *File → Import Notebook* with this file, *Session options*: **Accelerator GPU T4**, **Internet on**.

**First run:** `RESUME = False`. Downloads `WIKI_PARTS` of German Wikipedia, tokenizes them into shards (kept in the
**Output**), trains a new model.

**Every further run:** *Add Input → Your Work → Notebooks →* this notebook, `RESUME = True`, *Save & Run All*. Shards,
binary (unless the Rust code changed), model and AdamW state come from the input.


In [ ]:
# ── Settings ─────────────────────────────────────────────────────────────
REPO = "https://github.com/RodionKovalenko/rustCrates"
BRANCH = "feature/neural-network-rust"
WIKI_PARTS = "0-7"        # Wikipedia parquet parts (0..19), ~150 M tokens each; only used when no shards are in the input
STEPS = 20000            # optimizer steps of this run (the time limit may stop it earlier; the learning rate decays over them)
BATCH_SIZE = 16          # sequences of 512 tokens per step
MICRO_BATCH = 16         # sequences per forward/backward pass (halved automatically if the GPU runs out of memory)
LEARNING_RATE = 0.0003   # stable norm and gate since 4ca4494 (the earlier divergence came from them)
RESUME = False           # continue the model found in the notebook input
WINDOW = 256             # attention window of a new model (tokens each token sees back)
MAX_HOURS = 11.0
PRECISION = "f16"        # GEMM inputs: f16 (tensor cores, loss scaling) or f32

INPUT_ROOT = "/kaggle/input"
OUTPUT_ROOT = "/kaggle/working"
SOURCE_DIR = "/tmp/rustCrates"


In [ ]:
import glob, json, os, shutil, subprocess, sys, time, zipfile

def run(cmd, cwd=None, env=None):
    print("$", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True, cwd=cwd, env=env)

if subprocess.run("nvidia-smi", shell=True).returncode != 0:
    raise SystemExit("No GPU: set Session options -> Accelerator -> GPU T4 x2 (Save Version uses the notebook's accelerator), then run again.")

# Rust (the repository's rust-toolchain file selects the version on the first cargo call)
if not os.path.exists(os.path.expanduser("~/.cargo/bin/cargo")):
    run("curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal")
os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + os.pathsep + os.environ["PATH"]

# Source
if os.path.exists(SOURCE_DIR):
    run(f"git fetch --depth 200 origin {BRANCH} && git checkout -B {BRANCH} FETCH_HEAD", cwd=SOURCE_DIR)
else:
    run(f"git clone --depth 200 --branch {BRANCH} {REPO} {SOURCE_DIR}")
run("git log -1 --oneline", cwd=SOURCE_DIR)


In [ ]:
# cudarc loads the CUDA driver, cuBLAS and NVRTC at run time: make the toolkit libraries findable.
libs = sorted({os.path.dirname(p) for pattern in ("libcublas.so*", "libnvrtc.so*") for p in glob.glob(f"/usr/local/cuda*/**/{pattern}", recursive=True)})
os.environ["LD_LIBRARY_PATH"] = os.pathsep.join(libs + [os.environ.get("LD_LIBRARY_PATH", "")])
print("CUDA library dirs:", libs)

# Build (~10 minutes, no CUDA toolkit needed), or reuse the binary of an earlier output when the code is unchanged
features = "--features gpu-engine"
binary = os.path.join(SOURCE_DIR, "target", "release", "wav-transformer")
commit = subprocess.run("git rev-parse HEAD", shell=True, cwd=SOURCE_DIR, capture_output=True, text=True).stdout.strip()
BUILD_INPUTS = "src Cargo.toml Cargo.lock build.rs rust-toolchain"

def reusable_binary():
    for info_path in sorted(glob.glob(os.path.join(INPUT_ROOT, "**", "bin", "build.json"), recursive=True)):
        with open(info_path) as f:
            info = json.load(f)
        cached = os.path.join(os.path.dirname(info_path), "wav-transformer")
        if info.get("features") != features or not os.path.exists(cached):
            continue
        if info.get("commit") == commit or subprocess.run(f"git diff --quiet {info.get('commit')} HEAD -- {BUILD_INPUTS}", shell=True, cwd=SOURCE_DIR).returncode == 0:
            return cached, info
        print(f"Cached binary of {info.get('commit', '?')[:7]} is outdated")
    return None, None

cached, info = reusable_binary()
if cached:
    os.makedirs(os.path.dirname(binary), exist_ok=True)
    shutil.copy(cached, binary)
    os.chmod(binary, 0o755)
    print(f"Using the cached binary built from {info['commit'][:7]}; no build needed")
else:
    run(f"cargo build --release {features}", cwd=SOURCE_DIR)
    info = {"commit": commit, "features": features}

bin_dir = os.path.join(OUTPUT_ROOT, "bin")
os.makedirs(bin_dir, exist_ok=True)
shutil.copy(binary, os.path.join(bin_dir, "wav-transformer"))
with open(os.path.join(bin_dir, "build.json"), "w") as f:
    json.dump(info, f)


In [ ]:
# Pretraining data: token shards from an earlier output, or new ones from Wikipedia
def find_shards():
    for meta in sorted(glob.glob(os.path.join(INPUT_ROOT, "**", "pretrain", "meta.json"), recursive=True)):
        return os.path.dirname(meta)
    return None

DATA_DIR = find_shards()
if DATA_DIR:
    print("Using the token shards in", DATA_DIR)
    # Kept in the output as well, so that the next run (with this output as input) finds them again.
    shutil.copytree(DATA_DIR, os.path.join(OUTPUT_ROOT, "pretrain"), dirs_exist_ok=True)
else:
    DATA_DIR = os.path.join(OUTPUT_ROOT, "pretrain")
    run(f"pip install -q tokenizers pyarrow")
    run(f"python kaggle/prepare_pretrain.py --out {DATA_DIR} --parts {WIKI_PARTS} --download-dir /tmp/wiki_download", cwd=SOURCE_DIR)
    shutil.rmtree("/tmp/wiki_download", ignore_errors=True)
with open(os.path.join(DATA_DIR, "meta.json")) as f:
    meta = json.load(f)
print(f"{meta['train_tokens'] / 1e6:.0f} M training tokens, {meta['val_tokens'] / 1e6:.1f} M validation tokens")


In [ ]:
# Previous model (Add Input of an earlier output)
CHECKPOINT = "engine_lm"
storage = os.path.join(SOURCE_DIR, "STORAGE")
os.makedirs(storage, exist_ok=True)
previous = None
if RESUME:
    for path in sorted(glob.glob(os.path.join(INPUT_ROOT, "**", "model", CHECKPOINT), recursive=True)):
        previous = os.path.dirname(path)
        break
if previous:
    for name in os.listdir(previous):
        if name.startswith(CHECKPOINT):
            shutil.copy(os.path.join(previous, name), storage)
    print("Continuing the model from", previous)
else:
    print("Training a new model" + (" (no previous model in the input)" if RESUME else ""))
MODE = "resume" if previous else "new"


In [ ]:
# Pretrain; the full log is kept in the output as pretrain.log
env = dict(os.environ, TRAIN_MAX_HOURS=str(MAX_HOURS), MICRO_BATCH=str(MICRO_BATCH))
cmd = [binary, "engine-pretrain", DATA_DIR, str(STEPS), str(BATCH_SIZE), str(LEARNING_RATE), MODE, str(WINDOW), PRECISION]
print("$", " ".join(cmd), flush=True)
started = time.time()
with open(os.path.join(OUTPUT_ROOT, "pretrain.log"), "a") as log:
    proc = subprocess.Popen(cmd, cwd=SOURCE_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        log.write(line)
        log.flush()
        print(f"[{(time.time() - started) / 3600:5.2f} h] {line.rstrip()}", flush=True)
    proc.wait()
print("exit code", proc.returncode)


In [ ]:
# Output: the model (+ config, training state, validation history), a zip of it and the loss curve
output = os.path.join(OUTPUT_ROOT, "model")
os.makedirs(output, exist_ok=True)
for name in os.listdir(storage):
    if name.startswith(CHECKPOINT):
        shutil.copy(os.path.join(storage, name), output)
print("Saved the model to", output)

import csv
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
history = os.path.join(output, CHECKPOINT + ".history.csv")
rows = [r for r in csv.reader(open(history)) if len(r) >= 2] if os.path.exists(history) else []
if not rows:
    print("No validation results yet (see the training log above)")
else:
    steps, losses = [int(r[0]) for r in rows], [float(r[1]) for r in rows]
    fig, ax = plt.subplots(figsize=(10, 5.5), dpi=120)
    ax.plot(steps, losses, color="#e07b00", linewidth=2, label="Validation")
    best = min(range(len(losses)), key=losses.__getitem__)
    ax.scatter([steps[best]], [losses[best]], color="#e07b00", zorder=3)
    ax.annotate(f"best {losses[best]:.3f} (step {steps[best]})", (steps[best], losses[best]), textcoords="offset points", xytext=(0, 10), ha="center")
    ax.set_xlabel("Optimizer step"); ax.set_ylabel("Validation loss per token"); ax.set_title("language model (own engine)")
    ax.grid(alpha=0.3); ax.legend(); fig.tight_layout()
    fig.savefig(os.path.join(OUTPUT_ROOT, "loss_curve.png")); plt.show()
